# EEG-Twin — run the pipeline

CEBRA trajectory embedding → Transformer digital twin → paper figures.

**Runtime → Change runtime type → GPU** before running anything.

This notebook lives in the same Drive folder as the data. It reads the data
from there, clones the code from GitHub, and writes results back beside the
data. Run the cells top to bottom.

In [ ]:
#@title Settings { display-mode: "form" }

#@markdown Leave blank to use the folder this notebook is in.
DATA_FOLDER = ""  #@param {type:"string"}

#@markdown Twin training: 10 seeds is the paper setting. Lower it if the
#@markdown Colab session times out before it finishes.
TWIN_SEEDS = 10  #@param {type:"slider", min:1, max:10, step:1}

#@markdown A 4-patient, 1-epoch pass that checks the wiring in ~10 min.
#@markdown The numbers it produces are meaningless.
SMOKE_TEST = False  #@param {type:"boolean"}

#@markdown Code repository to clone.
REPO = "https://github.com/edilbertoamorim/ICARE_Twin.git"  #@param {type:"string"}

## 1 · Mount Drive and find the data

Two routes, picked automatically from what is in the folder:

| | Folder holds | What runs |
|---|---|---|
| **A** | `raw/protopnet/`, `raw/qeeg/`, `raw/offsets.csv` (54 GB) | the dataset build, then everything |
| **B** | `PPNet_data_train.npz`, `PPNet_data_test.npz` (1.6 GB) | everything after the build |

Both need `ICARE_clinical.csv`, and both give the same results — B just starts
later. Files may sit flat in the folder or mirror the repo's `data/` layout.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, glob

drive.mount('/content/drive')
MD = Path('/content/drive/MyDrive')

if DATA_FOLDER:
    SRC = MD / DATA_FOLDER
    if not SRC.is_dir():
        raise SystemExit(f'No such folder: {SRC}')
else:
    # Convenience only. Duplicate filenames across My Drive are common, so this
    # refuses to guess rather than silently picking one.
    hits = (glob.glob(f'{MD}/**/PPNet_data_train.npz', recursive=True) or
            glob.glob(f'{MD}/**/offsets.csv', recursive=True))
    folders = sorted({str(Path(h).parent) for h in hits})
    if not folders:
        raise SystemExit('Found neither PPNet_data_train.npz nor offsets.csv in My '
                         'Drive.\nSet DATA_FOLDER above to the folder you uploaded.')
    if len(folders) > 1:
        raise SystemExit('Several folders match — set DATA_FOLDER above to one of:\n  '
                         + '\n  '.join(f.replace(f'{MD}/', '') for f in folders))
    SRC = Path(folders[0])
    if SRC.name in ('dataset', 'raw'):
        SRC = SRC.parent

def find(name, *subs):
    """Check SRC and the named subfolders only. A recursive sweep would pick an
    arbitrary copy when the folder holds duplicates."""
    for d in ('',) + subs:
        q = SRC / d / name
        if q.is_file():
            return q
    return None

DATASET  = find('PPNet_data_train.npz', 'dataset', 'data/dataset')
OFFSETS  = find('offsets.csv', 'raw', 'data/raw')
CLINICAL = find('ICARE_clinical.csv', 'tables', 'data/tables')
ROUTE    = 'B' if DATASET else 'A' if OFFSETS else None

print(f'\nfolder   {SRC}')
if ROUTE is None:
    raise SystemExit(f'Nothing usable in {SRC}. Need PPNet_data_train.npz (route B), '
                     'or raw/offsets.csv + raw/protopnet/ + raw/qeeg/ (route A).')
if CLINICAL is None:
    raise SystemExit(f'ICARE_clinical.csv not found in {SRC} — both routes need it.')

if ROUTE == 'B':
    print('route    B — starting from the built dataset')
    print(f'dataset  {DATASET}')
else:
    RAW = OFFSETS.parent
    for sub in ('protopnet', 'qeeg'):
        if not (RAW / sub).is_dir():
            raise SystemExit(f'Route A also needs {RAW}/{sub}/')
    print('route    A — building the dataset from raw (adds hours; reads over Drive)')
    print(f'raw      {RAW}')
print(f'clinical {CLINICAL}')

## 2 · Install

Run this, **restart the session** when it says to, then run it again.

In [ ]:
import subprocess, sys, importlib.util
if not os.path.exists('/content/repo'):
    subprocess.run(['git', 'clone', '-q', REPO, '/content/repo'], check=True)
%cd /content/repo
!git pull -q 2>/dev/null
!pip install -q -r requirements.txt 2>&1 | grep -viE "dependency resolver|which is incompatible|^$" | head -3

if all(importlib.util.find_spec(m) for m in ('torch', 'cebra', 'plotly')):
    print('\nready')
else:
    print('\ninstalled — RESTART THE SESSION (Runtime > Restart session), then re-run this cell')

## 3 · Stage the data into the repo

In [ ]:
import shutil
%cd /content/repo
for d in ('data/dataset', 'data/tables'):
    Path(d).mkdir(parents=True, exist_ok=True)

shutil.copy(CLINICAL, 'data/tables/')
for n in ('split_train.csv', 'split_test.csv'):
    q = find(n, 'tables', 'data/tables')
    if q:
        shutil.copy(q, 'data/tables/')

if ROUTE == 'B':
    for n in ('PPNet_data_train.npz', 'PPNet_data_test.npz'):
        q = find(n, 'dataset', 'data/dataset')
        if q is None:
            raise SystemExit(f'{n} missing — route B needs both train and test.')
        if not Path('data/dataset', n).exists():
            print(f'copying {n} ({q.stat().st_size/1e9:.1f} GB) ...', flush=True)
            shutil.copy(q, 'data/dataset/')
    import numpy as np
    for s in ('train', 'test'):
        d = np.load(f'data/dataset/PPNet_data_{s}.npz', allow_pickle=True)
        print(f'  {s}: {len(np.unique(d["patient_ids"])):,} patients, '
              f'{len(d["patient_ids"]):,} segments')
else:
    # 54 GB — link it, never copy. Reads go straight to the Drive mount.
    if not Path('data/raw').exists():
        os.symlink(RAW, 'data/raw')
    print('linked   data/raw ->', RAW)
    for sub in ('protopnet', 'qeeg'):
        print(f'  {sub}: {len(list((RAW / sub).rglob("*")))} files')

print('\ntables:', ', '.join(sorted(q.name for q in Path('data/tables').iterdir())))

## 4 · Run

Every stage in order, skipping whatever is already built and rebuilding
anything whose inputs changed. Safe to re-run after a disconnect — it picks up
where it stopped.

Route A spends hours on the dataset build first. Route B skips straight to
CEBRA (~5 min on a T4). Twin training is the long pole either way and shows a
per-seed progress bar, so you can tell early whether it will finish.

In [ ]:
# Env goes on os.environ directly: the ! magic streams output to the notebook,
# subprocess.run() does not (its stdout bypasses Jupyter's display machinery).
if SMOKE_TEST:
    os.environ['CEBRA_SMOKE'] = os.environ['CEBRA_TWIN_SMOKE'] = '1'
    print('SMOKE TEST — 4 patients, 1 epoch. The numbers are meaningless.\n')
elif TWIN_SEEDS != 10:
    Path('sitecustomize.py').write_text(
        "import sys; sys.path.insert(0, 'src')\n"
        f"import config; config.TWIN_TRAIN.update(N_REF={TWIN_SEEDS}, "
        f"N_TWIN={TWIN_SEEDS}, N_ABL={TWIN_SEEDS}, N_CAL={TWIN_SEEDS})\n")
    os.environ['PYTHONPATH'] = '/content/repo'
    print(f'twin seeds set to {TWIN_SEEDS}\n')

!python -u run_all.py


## 5 · Save results back to the Drive folder

In [ ]:
DST = SRC / 'results'
DST.mkdir(exist_ok=True)

for sub in ('outputs', 'data/cebra', 'data/twin/handoffs'):
    s = Path(sub)
    if s.exists() and any(s.rglob('*')):
        d = DST / s.name
        shutil.rmtree(d, ignore_errors=True)
        shutil.copytree(s, d)
        print('saved', sub)

print(f'\n{DST}')
for p in sorted(DST.rglob('*')):
    if p.is_file():
        print(f'   {p.stat().st_size/1e6:8.1f} MB  {p.relative_to(DST)}')

## Results

`results/outputs/figures/` — each figure as interactive HTML, 600-dpi PNG and
vector PDF.

| | |
|---|---|
| `cebra/` | trajectory globes · twin in CEBRA space · prototype map |
| `twin/` | fig1-7 |
| `eval/` | confusion · CPC · centroid distances |

Open an HTML to rotate a globe — it prints the camera as a dict you can paste
into the script's `CAMERA` to lock that view for the static export.

## If it stops

**Session died.** Re-run cells 1-4. Finished stages are cached; you lose only
the stage that was running.

**Twin never finishes.** Drop `TWIN_SEEDS` to 2-3 and re-run cell 4. Figures
still build — the ensemble is just smaller.

**Something looks stale.** The runner rebuilds a stage when its inputs are
newer than its outputs. To force everything: `!python run_all.py --force`.